# Подсчёт досок и коротких досок — запуск в Google Colab

Блокнот обрабатывает видео с конвейера программой `measure_boards.py` на видеокарте Colab.

**Перед запуском:**
1. Меню **Среда выполнения → Сменить среду выполнения → T4 GPU** → Сохранить.
2. Положите видео на свой **Google Диск** (например, в папку `boards/videos`).
3. Заполните настройки в ячейке **1** и нажмите **Среда выполнения → Выполнить все**.

Результаты (`boards.csv`, `summary.json`, по желанию `annotated.mp4`) сохраняются **на Google Диск** —
всё, что лежит в самом Colab, пропадает при отключении.

## 1. Настройки

In [ ]:
#@title Настройки (заполните и запустите)
# Откуда взять код программы:
#   "git" — скачать из репозитория GitHub (для открытого репозитория);
#   "zip" — архив кода с Google Диска (для приватного: скачайте «Source code (zip)» со страницы Releases
#           и положите на Диск).
CODE_SOURCE = "git"  #@param ["git", "zip"]
REPO_URL = "https://github.com/OWNER/REPO.git"  #@param {type:"string"}
CODE_ZIP = "/content/drive/MyDrive/boards/code.zip"  #@param {type:"string"}

# Модель best.pt: пусто — скачать по ссылке из config.toml (model.url);
# иначе — путь к best.pt на Google Диске (для приватного репозитория скачайте его из Releases).
MODEL_ON_DRIVE = ""  #@param {type:"string"}

# Видео: один файл ИЛИ папка (тогда обрабатываются все .mp4 в ней по очереди).
VIDEO = "/content/drive/MyDrive/boards/videos/sample.mp4"  #@param {type:"string"}
# Куда сохранять результаты (на Google Диске).
RESULTS = "/content/drive/MyDrive/boards/results"  #@param {type:"string"}

# Видео с разметкой: большое (17 минут видео ~ 650 МБ) и медленнее. Для проверки — да, для подсчёта — нет.
MAKE_VIDEO = False  #@param {type:"boolean"}
# Обработать только кусок (секунды); 0 и 0 — всё видео.
START_S = 0  #@param {type:"number"}
END_S = 0  #@param {type:"number"}

## 2. Видеокарта и Google Диск

In [ ]:
import torch
if torch.cuda.is_available():
    print("Видеокарта:", torch.cuda.get_device_name(0))
else:
    print("ВНИМАНИЕ: видеокарты нет — будет считаться на процессоре (медленно).\n"
          "Меню: Среда выполнения → Сменить среду выполнения → T4 GPU, затем запустите блокнот заново.")

from google.colab import drive
drive.mount("/content/drive")

## 3. Код программы и библиотеки

In [ ]:
import os, shutil, zipfile, glob
from pathlib import Path

WORK = Path("/content/boards")
if WORK.exists():
    shutil.rmtree(WORK)

if CODE_SOURCE == "git":
    !git clone --depth 1 "{REPO_URL}" /content/boards_repo
    src = Path("/content/boards_repo")
else:
    with zipfile.ZipFile(CODE_ZIP) as z:
        z.extractall("/content/boards_zip")
    src = Path("/content/boards_zip")

# Папка программы — та, где лежит measure_boards.py (в архиве GitHub она вложена на уровень ниже).
found = sorted(src.rglob("measure_boards.py"))
assert found, "measure_boards.py не найден — проверьте REPO_URL или CODE_ZIP"
shutil.move(str(found[0].parent), WORK)
os.chdir(WORK)
print("Программа:", WORK)

# PyTorch, OpenCV и NumPy в Colab уже стоят; доставляем только Ultralytics нужной версии.
!pip -q install ultralytics==8.4.176

## 4. Модель (`models/best.pt`)

In [ ]:
Path("models").mkdir(exist_ok=True)
if MODEL_ON_DRIVE:
    shutil.copy(MODEL_ON_DRIVE, "models/best.pt")
    print("Модель взята с Диска:", MODEL_ON_DRIVE)
else:
    !python download_model.py

# Проверка настроек: модель на месте, контрольная сумма, на чём считается нейросеть.
!python settings.py

## 5. Обработка видео

In [ ]:
import subprocess, sys

videos = sorted(glob.glob(os.path.join(VIDEO, "*.mp4"))) if os.path.isdir(VIDEO) else [VIDEO]
assert videos and all(os.path.exists(v) for v in videos), f"Видео не найдено: {VIDEO}"
print("Видео к обработке:", *videos, sep="\n  ")

for v in videos:
    out = os.path.join(RESULTS, Path(v).stem)
    cmd = [sys.executable, "measure_boards.py", v, "--out", out]
    if not MAKE_VIDEO:
        cmd.append("--no-video")
    if START_S:
        cmd += ["--start", str(START_S)]
    if END_S:
        cmd += ["--end", str(END_S)]
    print("\n" + "=" * 70 + f"\n{Path(v).name}\n" + "=" * 70, flush=True)
    # построчный вывод хода работы (обычный subprocess.run в Colab печатать не будет)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         encoding="utf-8", errors="replace",
                         env={**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8"})
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise SystemExit(f"Ошибка при обработке {v} — см. сообщения выше")

## 6. Результаты

In [ ]:
import json
import pandas as pd

for v in videos:
    out = Path(RESULTS) / Path(v).stem
    s = json.load(open(out / "summary.json", encoding="utf-8"))
    print(f"\n{Path(v).name}: досок {s['boards']}, выбросов {s['outliers']}, не измерено {s['not_measured']}, "
          f"не доехали до конца видео {s['unfinished']}  (обработка {s['processing_s'] / 60:.1f} мин "
          f"на {s['video_s'] / 60:.1f} мин видео)")
    display(pd.read_csv(out / "boards.csv", sep=";", encoding="utf-8-sig"))
print("\nФайлы результатов — на Google Диске:", RESULTS)

## Если что-то пошло не так

| Сообщение | Что делать |
|---|---|
| `видеокарты нет` | Среда выполнения → Сменить среду выполнения → T4 GPU, запустить заново |
| `could not read Username` при `git clone` | репозиторий приватный — `CODE_SOURCE = "zip"` (архив кода из Releases на Диске) |
| `не указана ссылка model.url` | указать `MODEL_ON_DRIVE` — путь к `best.pt` на Диске |
| `скачался не тот файл (контрольная сумма ...)` | ссылка в `config.toml` ведёт не на ту версию модели |
| `Видео не найдено` | проверить путь `VIDEO`: на Диске он начинается с `/content/drive/MyDrive/` |
| сессия отключилась посреди работы | бесплатный Colab отключается после ~90 минут без активности; готовые видео уже на Диске — запустите остальные |

Подробно о программе — `README.md` в репозитории.